In [1]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


In [2]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!ollama --version

In [4]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


In [5]:
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [6]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-19T05:08:26.392232624Z","response":"Hello!","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,0],"total_duration":110973848198,"load_duration":60855572534,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":50074499000,"eval_count":3,"eval_duration":40960000}

In [8]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')
print(".env created. Key length:", len(key))

.env created. Key length: 53


In [9]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [10]:
!pip install -q -r requirements.txt
from google import genai
print("google-genai imported OK")

google-genai imported OK


In [19]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests


class LLMProvider(ABC):
    @abstractmethod
    def generate(self, prompt):
        pass


class GeminiProvider(LLMProvider):
    def __init__(self, api_key, model="gemini-3.5-flash-lite"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


class OllamaProvider(LLMProvider):
    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


class LLMClient:
    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Overwriting llm_client.py


In [20]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)
elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")
else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)
prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)

Overwriting main.py


In [13]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient


class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):
    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)
        result = client.generate("Explain AI")
        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()

Writing test_client.py


In [21]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: typeguard-4.6.0, platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.20s ===============================


In [22]:
%run main.py

Choose provider (gemini/ollama): gemini
Enter your question: what is ai 

LLM Response:
**AI (Artificial Intelligence)** refers to the ability of a computer, machine, or software to mimic human intelligence. 

Instead of just following rigid, pre-programmed instructions, an AI system can **learn from data, recognize patterns, solve problems, and make decisions.**

Here is a simple breakdown of what AI is, how it works, and how it is used:

---

### 1. How Does AI Work?
Traditional computer programs work like a recipe: *“If the user clicks this button, do that.”* 

AI works more like a human child learning:
1. **Data:** You feed the computer vast amounts of information (e.g., millions of photos of cats and dogs).
2. **Algorithms (Machine Learning):** The computer looks for patterns on its own (e.g., recognizing that cats usually have pointed ears and whiskers).
3. **Prediction/Action:** When shown a *new* picture, the AI guesses whether it is a cat or a dog based on the patterns it lear